# AS Tagging Toolkit - Example Notebook

This notebook demonstrates the usage of preset composite tags and the API for the AS Tagging Toolkit.

## Data Loading Modes

The toolkit supports **two modes** for loading AS feature snapshots:

1. **Online Mode** - Load data directly from HuggingFace (`zchen798/as_feature_snapshot`)
   - Requires HuggingFace token for private repository access
   - Automatic caching for faster subsequent loads
   
2. **Offline Mode** - Load data from local files
   - Requires downloading snapshot files in advance
   - Useful for air-gapped environments or custom datasets

---

**Implemented Preset Composite Tags:**
1. Anycast
2. Tranco 10k Host
3. IPv6 Only
4. No Eyeball
5. No Transit
6. Sibling Transit
7. Public Transit
8. Any Presence
9. Domestic
10. Major Access
11. Global Transit Nth-ranked

---
## Setup Option 1: Online Mode (HuggingFace)

Use `OnlineSnapshotProvider` to load data directly from HuggingFace. This is the recommended approach for most users.

**Requirements:**
- Install `huggingface_hub`: `pip install huggingface_hub`
- HuggingFace token with access to the private repository

In [ ]:
from as_tagging import ASTagging, OnlineSnapshotProvider
import os

# Disable tqdm progress bars to avoid ipywidget rendering issues
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

# Option 1a: Provide token directly
# HF_TOKEN = "your_huggingface_token_here"
# online_provider = OnlineSnapshotProvider(token=HF_TOKEN)

# Option 1b: Use HF_TOKEN environment variable (recommended)
# Set HF_TOKEN in your environment before running
online_provider = OnlineSnapshotProvider(token=os.environ.get("HF_TOKEN"))

# List available snapshots from HuggingFace
print("Available snapshots:", online_provider.list_snapshots())

In [ ]:
# Get snapshot info before loading
DATE = "2025-12"
info = online_provider.get_snapshot_info(DATE)
if info:
    print(f"Snapshot info for {DATE}:")
    print(f"  - Number of ASNs: {info.get('num_asns')}")
    print(f"  - Schema version: {info.get('schema_version')}")
    print(f"  - Source dates (sample): {list(info.get('source_dates', {}).items())[:3]}")

In [ ]:
# Initialize ASTagging with Online provider
# First load may download ~15MB; use_cache=False re-downloads the snapshot tarball and re-extracts instead of reusing ~/.cache/as_tagging/<DATE>/.
tagger = ASTagging(snapshot_provider=online_provider, date=DATE, use_cache=False)

print(f"Loaded {len(tagger.atomic_tags)} ASNs from HuggingFace")
print(f"Available composite tags: {list(tagger.tag_expressions.keys())}")

---
## Setup Option 2: Offline Mode (Local Files)

Use `OfflineSnapshotProvider` to load data from local files. Useful when you have pre-downloaded snapshots.

**Requirements:**
- Download snapshot files from Zenodo or other sources
- Directory structure with `index.json` and snapshot files (.tar.gz)

In [4]:
from as_tagging import ASTagging, OfflineSnapshotProvider

# Initialize with OfflineSnapshotProvider
DATA_PATH = fr"../as_feature_zenodo" # Update to your path
# Modify to your path
DATE = "2026-01"

offline_provider = OfflineSnapshotProvider(DATA_PATH)

# List available local snapshots
print("Available local snapshots:", offline_provider.list_snapshots())

# Initialize ASTagging
# use_cache=False re-extracts from the local .tar.gz instead of reusing a possibly stale ~/.cache/as_tagging/<DATE>/.
tagger = ASTagging(snapshot_provider=offline_provider, date=DATE, use_cache=False)

print(f"Loaded {len(tagger.atomic_tags)} ASNs from local files")
print(f"Available composite tags: {list(tagger.tag_expressions.keys())}")

Available local snapshots: ['2024-08', '2024-09', '2024-10', '2024-11', '2024-12', '2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03', '2026-04']
Loaded 120050 ASNs from local files
Available composite tags: ['Anycast', 'Tranco 10k Host', 'IPv6 Only', 'No Eyeball', 'No Transit', 'Sibling Transit', 'Public Transit', 'Any Presence', 'Domestic', 'Major Access']


## Basic API Usage

### ListTags - View all tags for an ASN

In [6]:
# List all tags for Georgia Tech's AS (AS2637)
tags = tagger.ListTags(3356)
print(f"Atomic tags: {len(tags['Atomic'])}")
print(f"Composite tags: {len(tags['Composite'])}")
print(f"\nComposite tags for AS2637: {tags['Composite']}")

Atomic tags: 97
Composite tags: 11

Composite tags for AS2637: {'Global Transit Nth-ranked': 1, 'Anycast': True, 'Tranco 10k Host': True, 'IPv6 Only': False, 'No Eyeball': False, 'No Transit': False, 'Sibling Transit': False, 'Public Transit': True, 'Any Presence': ['GR', 'SI', 'SE', 'JP', 'RU', 'LV', 'AT', 'SK', 'UM', 'SV', 'IL', 'TH', 'PH', 'TR', 'HT', 'TW', 'IT', 'NG', 'NO', 'NZ', 'LT', 'UA', 'IN', 'MC', 'BR', 'CL', 'MN', 'AE', 'GB', 'CR', 'BE', 'MY', 'GT', 'AU', 'HR', 'ID', 'PA', 'CZ', 'PT', 'ES', 'ZA', 'DE', 'VI', 'MP', 'LU', 'PL', 'PE', 'AO', 'NL', 'CA', 'CY', 'FR', 'AR', 'GU', 'EE', 'CH', 'IE', 'RO', 'NI', 'US', 'HU', 'KR', 'BG', 'CO', 'EC', 'DO', 'SG', 'RS', 'MX', 'HN', 'FI', 'UNKNOWN', 'HK', 'DK', 'PR', 'VE', 'BT'], 'Domestic': ['US'], 'Major Access': []}


In [14]:
# print(tags['Atomic'])
for tag in tags['Atomic']:
    if "caida" in tag:
        print(tag)
    if "apnic" in tag:
        print(tag)
    # if "merit" in tag:
    #     print(tag)
    if "offnet" in tag:
        print(tag)
    # if "censys" in tag:
    #     print(tag)
    if "maxmind" in tag:
        print(tag)
    if "mlab" in tag:
        print(tag)

apnic-eyeball_cc_cnt
apnic-eyeball_eyeball_cnt
apnic-eyeball_gini
apnic-eyeball_top_cc(frac)
apnic-eyeball_top_cc(num)
apnic-eyeball_top_frac
caida-asrel_cone_/24_cnt
caida-asrel_cone_/64_cnt
caida-asrel_cone_as_cnt
caida-asrel_cone_as_list
caida-asrel_customer_cnt
caida-asrel_customer_list
caida-asrel_peer_cnt
caida-asrel_peer_list
caida-asrel_provider_cnt
caida-asrel_provider_list
caida-itdk_cc_cnt
caida-itdk_cc_cnt_v4
caida-itdk_cc_cnt_v6
caida-itdk_gini
caida-itdk_gini_v4
caida-itdk_gini_v6
caida-itdk_router_cnt
caida-itdk_router_cnt_v4
caida-itdk_router_cnt_v6
caida-itdk_topcc
caida-itdk_topcc_v4
caida-itdk_topcc_v6
caida-itdk_topfrac
caida-itdk_topfrac_v4
caida-itdk_topfrac_v6
hg-offnet_v4addr_cnt
maxmind-geolite2_cc_v4_cnt
maxmind-geolite2_cc_v4_dict
maxmind-geolite2_cc_v6_cnt
maxmind-geolite2_cc_v6_dict
maxmind-geolite2_gini_v4
maxmind-geolite2_gini_v6
maxmind-geolite2_topcc_v4
maxmind-geolite2_topcc_v6
maxmind-geolite2_topfrac_v4
maxmind-geolite2_topfrac_v6
mlab-ndt_/24_cnt
ml

In [ ]:
tags = tagger.ListTags(142650)
print(f"Atomic tags: {len(tags['Atomic'])}")
print(f"Composite tags: {len(tags['Composite'])}")
print(f"\nComposite tags for AS142650: {tags['Composite']}")

### FetchTag - Retrieve specific tag values

In [ ]:
# Fetch atomic tag
print(f"AS2637 IPv4 /24 count: {tagger.FetchTag('pfx2as_/24_cnt', asns=2637)}")
print(f"AS2637 customer ASes: {tagger.FetchTag('caida-asrel_cone_as_list', asns=2637)}")

In [ ]:
# Fetch composite tag for multiple ASNs
print(f"Domestic for AS2637 and AS12389: {tagger.FetchTag('Domestic', asns=[2637, 12389])}")

In [12]:
non_eyeball_ases = tagger.FetchTag("No Eyeball")

In [15]:
# AS degree = customers + peers + providers (CAIDA AS-relationship counts)
high_degree_asns = [
    asn
    for asn, tags in tagger.atomic_tags.items()
    if (
        float(tags.get("caida-asrel_customer_cnt") or 0)
        + float(tags.get("caida-asrel_peer_cnt") or 0)
        + float(tags.get("caida-asrel_provider_cnt") or 0)
    ) > 100
]
print(f"ASes with AS degree > 100: {len(high_degree_asns)}")

ASes with AS degree > 100: 2060


In [16]:
import json
with open("./data/no_eyeball_asns.json", "w") as f:
    json.dump(non_eyeball_ases, f)
with open("./data/high_degree_asns.json", "w") as f:
    json.dump(high_degree_asns, f)

In [ ]:
eyeball_asns = tagger.ListASNsWithoutTag("No Eyeball", treat_false_as_missing=True)
from collections import Counter

eyeball_asn_rir = tagger.FetchTag("delegation_rir", eyeball_asns)
value_counts = Counter(eyeball_asn_rir.values())
print(value_counts)

# ASNs with inferred eyeballs but no AS2Org siblings (standalone org in that snapshot)
eyeball_single_asns = [
    asn
    for asn in eyeball_asns
    if float(tagger.atomic_tags.get(asn, {}).get("apnic-eyeball_eyeball_cnt") or 0) > 0
    and float(tagger.atomic_tags.get(asn, {}).get("inetintel-as2org_sibling_cnt") or 0) == 0
]
eyeball_single_asn_rir = tagger.FetchTag("delegation_rir", eyeball_single_asns)
print(Counter(eyeball_single_asn_rir.values()))

In [3]:
no_transit_asns = tagger.FetchTag("No Transit")
sib_transit_asns = tagger.FetchTag("Sibling Transit")
pub_transit_asns = tagger.FetchTag("Public Transit")
print(len(no_transit_asns), len(sib_transit_asns), len(pub_transit_asns))

107639 755 11656


In [6]:
import json
import os
os.makedirs("./data/transit_asns", exist_ok=True)
with open("./data/transit_asns/no_transit_asns.json", "w") as f:
    json.dump(no_transit_asns, f)
with open("./data/transit_asns/sib_transit_asns.json", "w") as f:
    json.dump(sib_transit_asns, f)
with open("./data/transit_asns/pub_transit_asns.json", "w") as f:
    json.dump(pub_transit_asns, f)


In [ ]:
rir_eyeball_asn = {}
for asn, rir in eyeball_asn_rir.items():
    rir_eyeball_asn.setdefault(rir, []).append(asn)
print(rir_eyeball_asn)

rir_eyeball_single_asn = {}
for asn, rir in eyeball_single_asn_rir.items():
    rir_eyeball_single_asn.setdefault(rir, []).append(asn)
print(rir_eyeball_single_asn)
### Help - Get tag description

In [ ]:
import json
with open('data/rir_eyeball_asn.json', 'w') as f:
    json.dump(rir_eyeball_asn, f)
with open('data/rir_eyeball_single_asn.json', 'w') as f:
    json.dump(rir_eyeball_single_asn, f)
### Help - Get tag description

### Help - Get tag description

In [ ]:
tagger.Help("merit_/24_cnt")

In [ ]:
tagger.Help("hg-offnet_v4addr_cnt")

In [ ]:
tagger.Help("Domestic")

---
## Composite Tag Testing

Below we test each preset composite tag with examples.

### 1. Anycast

**Definition:** AS uses anycast for at least one IP address.

In [ ]:
# Test Anycast tag for known CDN ASNs
test_asns = [13335, 15169, 7018, 2637]  # Cloudflare, Google, AT&T, Georgia Tech
print("=== Anycast Tag Test ===")
for asn in test_asns:
    result = tagger.FetchTag("Anycast", asns=asn)
    print(f"  AS{asn}: {result}")

In [ ]:
# First 5 ASNs with Anycast
anycast_asns = [(asn, tags.get('Anycast')) 
                for asn, tags in tagger.composite_tags.items() 
                if tags.get('Anycast', False)][:5]
print("First 5 Anycast ASNs:")
for asn, val in anycast_asns:
    print(f"  AS{asn}: {val}")

total = sum(1 for asn, tags in tagger.composite_tags.items() if tags.get('Anycast', False))
print(f"\nTotal Anycast ASNs: {total}")

### 2. Tranco 10k Host

**Definition:** AS hosts domains in the Tranco top domains list.

In [ ]:
# Test for well-known hosting ASNs
test_asns = [13335, 15169, 16509, 2637]  # Cloudflare, Google, Amazon, Georgia Tech
print("=== Tranco 10k Host Tag Test ===")
for asn in test_asns:
    result = tagger.FetchTag("Tranco 10k Host", asns=asn)
    print(f"  AS{asn}: {result}")

In [ ]:
# First 5 ASNs hosting Tranco domains
tranco_asns = [(asn, tags.get('Tranco 10k Host')) 
               for asn, tags in tagger.composite_tags.items() 
               if tags.get('Tranco 10k Host', False)][:5]
print("First 5 Tranco 10k Host ASNs:")
for asn, val in tranco_asns:
    print(f"  AS{asn}: {val}")

total = sum(1 for asn, tags in tagger.composite_tags.items() if tags.get('Tranco 10k Host', False))
print(f"\nTotal Tranco 10k Host ASNs: {total}")

### 3. IPv6 Only

**Definition:** AS originates IPv6 prefixes but no IPv4 prefixes.

In [ ]:
# First 5 IPv6-only ASNs
ipv6_only_asns = [(asn, tags.get('IPv6 Only')) 
                  for asn, tags in tagger.composite_tags.items() 
                  if tags.get('IPv6 Only', False)][:5]
print("First 5 IPv6 Only ASNs:")
for asn, val in ipv6_only_asns:
    v6_cnt = tagger.FetchTag('pfx2as_/64_cnt', asns=asn)
    v4_cnt = tagger.FetchTag('pfx2as_/24_cnt', asns=asn)
    print(f"  AS{asn}: IPv6 Only = {val}, /64_cnt = {v6_cnt}, /24_cnt = {v4_cnt}")

total = sum(1 for asn, tags in tagger.composite_tags.items() if tags.get('IPv6 Only', False))
print(f"\nTotal IPv6 Only ASNs: {total}")

### 4. No Eyeball

**Definition:** AS has no inferred eyeballs (end users).

In [ ]:
# Test for known eyeball ASNs (these should be False)
test_asns = [7018, 7922, 13335, 2637]  # AT&T, Comcast (have eyeballs), Cloudflare, Georgia Tech
print("=== No Eyeball Tag Test (ASNs with eyeballs) ===")
for asn in test_asns:
    no_eyeball = tagger.FetchTag("No Eyeball", asns=asn)
    eyeball_cnt = tagger.FetchTag('apnic-eyeball_eyeball_cnt', asns=asn)
    print(f"  AS{asn}: No Eyeball = {no_eyeball}, eyeball_cnt = {eyeball_cnt:.0f}")

In [ ]:
# Show first 5 ASNs where No Eyeball = True (infrastructure/enterprise ASNs)
no_eyeball_asns = [(asn, tags.get('No Eyeball')) 
                   for asn, tags in tagger.composite_tags.items() 
                   if tags.get('No Eyeball', False)][:5]
print("=== No Eyeball = True (infrastructure/enterprise ASNs) ===")
for asn, val in no_eyeball_asns:
    eyeball_cnt = tagger.FetchTag('apnic-eyeball_eyeball_cnt', asns=asn)
    print(f"  AS{asn}: No Eyeball = {val}, eyeball_cnt = {eyeball_cnt}")

In [ ]:
# Stats
no_eyeball_count = sum(1 for asn, tags in tagger.composite_tags.items() if tags.get('No Eyeball', False))
has_eyeball_count = len(tagger.composite_tags) - no_eyeball_count
print(f"No Eyeball ASNs: {no_eyeball_count}")
print(f"Has Eyeball ASNs: {has_eyeball_count}")

### 5. No Transit

**Definition:** AS does not provide IP transit to any other AS (has no customers).

In [ ]:
# Test for known transit providers (these should be False)
test_asns = [3356, 174, 2914, 2637]  # Level3, Cogent, NTT (transit), Georgia Tech
print("=== No Transit Tag Test (transit providers) ===")
for asn in test_asns:
    no_transit = tagger.FetchTag("No Transit", asns=asn)
    customer_cnt = tagger.FetchTag('caida-asrel_customer_cnt', asns=asn)
    print(f"  AS{asn}: No Transit = {no_transit}, customer_cnt = {customer_cnt}")

In [ ]:
# Show first 5 ASNs where No Transit = True (stub networks)
no_transit_asns = [(asn, tags.get('No Transit')) 
                   for asn, tags in tagger.composite_tags.items() 
                   if tags.get('No Transit', False)][:5]
print("=== No Transit = True (stub networks with no downstream) ===")
for asn, val in no_transit_asns:
    customer_cnt = tagger.FetchTag('caida-asrel_customer_cnt', asns=asn)
    print(f"  AS{asn}: No Transit = {val}, customer_cnt = {customer_cnt}")

In [ ]:
# Stats
no_transit_count = sum(1 for asn, tags in tagger.composite_tags.items() if tags.get('No Transit', False))
print(f"No Transit ASNs: {no_transit_count}")
print(f"Percentage: {no_transit_count/len(tagger.composite_tags)*100:.1f}%")

### 6. Sibling Transit

**Definition:** AS only provides transit to sibling ASes (same organization).

In [ ]:
# Find ASNs with Sibling Transit
sibling_transit_asns = [(asn, tags.get('Sibling Transit')) 
                        for asn, tags in tagger.composite_tags.items() 
                        if tags.get('Sibling Transit', False)][:5]
print("First 5 Sibling Transit ASNs:")
for asn, val in sibling_transit_asns:
    customer_cnt = tagger.FetchTag('caida-asrel_customer_cnt', asns=asn)
    print(f"  AS{asn}: Sibling Transit = {val}, customer_cnt = {customer_cnt}")

total = sum(1 for asn, tags in tagger.composite_tags.items() if tags.get('Sibling Transit', False))
print(f"\nTotal Sibling Transit ASNs: {total}")

### 7. Public Transit

**Definition:** AS provides IP transit to non-sibling ASes.

In [ ]:
# Test for known transit providers
test_asns = [3356, 174, 2914, 6939]  # Level3, Cogent, NTT, Hurricane Electric
print("=== Public Transit Tag Test ===")
for asn in test_asns:
    public_transit = tagger.FetchTag("Public Transit", asns=asn)
    customer_cnt = tagger.FetchTag('caida-asrel_customer_cnt', asns=asn)
    print(f"  AS{asn}: Public Transit = {public_transit}, customer_cnt = {customer_cnt}")

In [ ]:
# Stats
public_transit_count = sum(1 for asn, tags in tagger.composite_tags.items() if tags.get('Public Transit', False))
print(f"Public Transit ASNs: {public_transit_count}")

### 8. Any Presence (Geolocation)

**Definition:** Returns list of country codes where AS has any IP presence.

In [ ]:
# Test for multinational ASNs
test_asns = [15169, 13335, 7018, 12389, 2637]  # Google, Cloudflare, AT&T, Rostelecom, Georgia Tech
print("=== Any Presence Tag Test ===")
for asn in test_asns:
    presence = tagger.FetchTag("Any Presence", asns=asn)
    print(f"  AS{asn}: {len(presence) if presence else 0} countries - {presence[:5] if presence else None}...")

In [ ]:
# Count ASNs with presence in specific countries
as2ccs = tagger.FetchTag("Any Presence")
country_counts = {"US": 0, "CN": 0, "RU": 0, "DE": 0, "JP": 0}
for asn, ccs in as2ccs.items():
    for cc in country_counts:
        if cc in ccs:
            country_counts[cc] += 1
print("ASNs with presence in each country:")
for cc, count in country_counts.items():
    print(f"  {cc}: {count}")

### 9. Domestic (Geolocation)

**Definition:** AS has >= 2/3 of its IP addresses geolocated in a country.

In [ ]:
# Test for known domestic ASNs
test_asns = [12389, 7018, 2637, 15169]  # Rostelecom (RU), AT&T (US), Georgia Tech (US), Google (multinational)
print("=== Domestic Tag Test ===")
for asn in test_asns:
    domestic = tagger.FetchTag("Domestic", asns=asn)
    print(f"  AS{asn}: Domestic in {domestic}")

In [ ]:
# Count domestic ASNs per country (sample countries)
as2ccs = tagger.FetchTag("Domestic")
country_counts = {"US": 0, "CN": 0, "RU": 0, "DE": 0, "JP": 0, "BR": 0, "IN": 0}
for asn, ccs in as2ccs.items():
    for cc in country_counts:
        if cc in ccs:
            country_counts[cc] += 1
print("Domestic ASNs per country:")
for cc, count in sorted(country_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {cc}: {count}")

### 10. Major Access (Geolocation)

**Definition:** AS originates >= 5% of a country's globally routed IP addresses.

In [ ]:
# Test for known major access providers
test_asns = [12389, 7018, 7922, 2637]  # Rostelecom (RU), AT&T (US), Comcast (US), Georgia Tech
print("=== Major Access Tag Test ===")
for asn in test_asns:
    major = tagger.FetchTag("Major Access", asns=asn)
    print(f"  AS{asn}: Major Access in {major}")

In [ ]:
# Show first 10 Major Access ASNs
major_access_asns = [(asn, tags.get('Major Access')) 
                     for asn, tags in tagger.composite_tags.items() 
                     if tags.get('Major Access') and len(tags.get('Major Access', [])) > 0][:10]
print("First 10 Major Access ASNs:")
for asn, countries in major_access_asns:
    print(f"  AS{asn}: {countries}")

total = sum(1 for asn, tags in tagger.composite_tags.items() 
            if tags.get('Major Access') and len(tags.get('Major Access', [])) > 0)
print(f"\nTotal Major Access ASNs: {total}")

### 11. Global Transit Nth-ranked

**Definition:** Global transit significance ranking using Borda count aggregation over 5 transit metrics.

In [ ]:
# Test for known transit providers
test_asns = [3356, 1299, 174, 2914, 6939, 7018, 15169]  # Level3, Arelion, Cogent, NTT, HE, AT&T, Google
print("=== Global Transit Nth-ranked Test ===")
for asn in test_asns:
    rank = tagger.FetchTag("Global Transit Nth-ranked", asns=asn)
    print(f"  AS{asn}: Rank {rank}")

In [ ]:
# Show top 20 global transit providers
ranked_asns = [(asn, tags.get('Global Transit Nth-ranked')) 
               for asn, tags in tagger.composite_tags.items() 
               if tags.get('Global Transit Nth-ranked')]
ranked_asns.sort(key=lambda x: x[1])

print("Top 20 Global Transit Providers (Borda Count Ranking):")
for asn, rank in ranked_asns[:20]:
    print(f"  Rank {rank:3d}: AS{asn}")

print(f"\nTotal ranked ASNs: {len(ranked_asns)}")

---
## Custom Tag Assignment

Users can define their own composite tags using `AssignTag`.

In [ ]:
# Define custom tag: Large IPv4 AS (> 0.1 million /24s)
tagger.AssignTag(
    tag_name="Large IPv4 AS",
    expression=lambda tags: (tags.get('pfx2as_/24_cnt', 0) or 0) > 100000
)

# Test
large_asns = [(asn, tags.get('Large IPv4 AS')) 
              for asn, tags in tagger.composite_tags.items() 
              if tags.get('Large IPv4 AS', False)][:10]
print("Large IPv4 ASNs (> 100k /24s):")
for asn, val in large_asns:
    v4_cnt = tagger.FetchTag('pfx2as_/24_cnt', asns=asn)
    print(f"  AS{asn}: {v4_cnt:,.0f} /24s")

---
## Customizing Preset Tag Thresholds

Some preset tags have default thresholds that can be customized by passing kwargs to `FetchTag`.

### Example 1: Domestic with different thresholds

The default Domestic threshold is 2/3 (66.7%). Let's see how results change with stricter thresholds.

In [ ]:
# Compare AS12389 (Rostelecom) with different Domestic thresholds
asn = 12389
print(f"=== Domestic threshold comparison for AS{asn} ===")

for threshold in [0.5, 0.66, 0.8, 0.9, 0.95]:
    # Use FetchTag with custom threshold
    result = tagger.FetchTag('Domestic', asns=asn, threshold=threshold)
    print(f"  threshold={threshold:.2f}: Domestic in {result}")

In [ ]:
# Count how many ASNs are domestic in Russia with different thresholds
print("=== Domestic ASNs in Russia (RU) with different thresholds ===")

all_asns = list(tagger.atomic_tags.keys())
for threshold in [0.5, 0.66, 0.8, 0.9, 0.95, 1.0]:
    domestic_map = tagger.FetchTag("Domestic", asns=all_asns, threshold=threshold)
    count = sum(1 for v in domestic_map.values() if v and "RU" in v)
    print(f"  threshold={threshold:.2f}: {count} ASNs domestic in RU")

### Example 2: Major Access with different thresholds

The default Major Access threshold is 5%. Change it to identify smaller or larger access networks.

In [ ]:
# Major Access with different thresholds
# Since Major Access requires country_sums, we use a workaround

import json

def get_major_access_countries(tags, threshold, country_sums):
    """Manually apply different thresholds"""
    v4_dict = tags.get('maxmind-geolite2_cc_v4_dict', {})
    if isinstance(v4_dict, str):
        try:
            v4_dict = json.loads(v4_dict)
        except:
            return []
    if not v4_dict:
        return []
    return [cc for cc, count in v4_dict.items() 
            if country_sums.get(cc, 0) > 0 and count / country_sums[cc] >= threshold]

# Test AS7018 (AT&T) with different thresholds
asn = 7018
tags = tagger.atomic_tags.get(asn, {})
country_sums = tagger.country_sums  # Access pre-computed country sums

print(f"=== Major Access threshold comparison for AS{asn} ===")
for threshold in [0.01, 0.02, 0.05, 0.10, 0.15]:
    result = get_major_access_countries(tags, threshold, country_sums)
    print(f"  threshold={threshold:.2f}: Major Access in {result}")

### Example 3: Define your own domestic-like tag with stricter criteria

In [ ]:
# Define "Purely Domestic" - AS with >95% of IPs in one country
import json

def purely_domestic(tags):
    v4_dict = tags.get('maxmind-geolite2_cc_v4_dict', {})
    if isinstance(v4_dict, str):
        try:
            v4_dict = json.loads(v4_dict)
        except:
            return []
    if not v4_dict:
        return []
    total = sum(v4_dict.values())
    if total == 0:
        return []
    return [cc for cc, count in v4_dict.items() if count / total >= 0.95]

tagger.AssignTag('Purely Domestic', purely_domestic)

# Count purely domestic ASNs per country
purely_domestic_asns = [(asn, tags.get('Purely Domestic')) 
                        for asn, tags in tagger.composite_tags.items() 
                        if tags.get('Purely Domestic') and len(tags.get('Purely Domestic', [])) > 0]

country_counts = {}
for asn, ccs in purely_domestic_asns:
    for cc in ccs:
        country_counts[cc] = country_counts.get(cc, 0) + 1

print("Purely Domestic (>95%) ASNs per country (top 10):")
for cc, count in sorted(country_counts.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  {cc}: {count}")

print(f"\nTotal Purely Domestic ASNs: {len(purely_domestic_asns)}")

---
## Summary Statistics

In [ ]:
# Count all preset composite tags
preset_tags = [
    "Anycast", "Tranco 10k Host", "IPv6 Only", "No Eyeball",
    "No Transit", "Sibling Transit", "Public Transit",
    "Any Presence", "Domestic", "Major Access", "Global Transit Nth-ranked"
]

print("=" * 50)
print("Composite Tag Summary")
print("=" * 50)

for tag in preset_tags:
    if tag in ["Any Presence", "Domestic", "Major Access"]:
        # List-valued tags
        count = sum(1 for asn, tags in tagger.composite_tags.items() 
                    if tags.get(tag) and len(tags.get(tag, [])) > 0)
    elif tag == "Global Transit Nth-ranked":
        # Numeric rank
        count = sum(1 for asn, tags in tagger.composite_tags.items() 
                    if tags.get(tag) is not None)
    else:
        # Boolean tags
        count = sum(1 for asn, tags in tagger.composite_tags.items() 
                    if tags.get(tag, False))
    print(f"{tag:30s}: {count:,} ASNs")

print("=" * 50)
print(f"Total ASNs: {len(tagger.atomic_tags):,}")